---
# Document Chunking Engine

Converts processed Markdown files into semantically meaningful chunks for RAG.
- Respects document structure (sections, paragraphs)
- Preserves metadata and context
- Optimizes chunk size for embedding models
- Prepares data for vector database ingestion

In [ ]:
import os
import json
import re
from pathlib import Path
from typing import Dict, List, Tuple, Optional
from dataclasses import dataclass
import tiktoken
from tqdm import tqdm

@dataclass
class ChunkMetadata:
    document_id: str
    chunk_id: str
    section: Optional[str]
    chunk_index: int
    total_chunks: int
    source_file: str
    paper_title: str
    authors: List[str]
    doi: str
    publication_date: str

class DocumentChunker:
    
    def __init__(self, 
                 chunk_size: int = 500,
                 chunk_overlap: int = 50,
                 min_chunk_size: int = 100):
        self.chunk_size = chunk_size
        self.chunk_overlap = chunk_overlap
        self.min_chunk_size = min_chunk_size
        
        try:
            self.tokenizer = tiktoken.get_encoding("cl100k_base")  # GPT-4 tokenizer, can try others too, baad mai lets see
        except:
            self.tokenizer = None
            print("Warning: tiktoken not available, using word count approximation")
    
    def count_tokens(self, text: str) -> int:
        if self.tokenizer:
            return len(self.tokenizer.encode(text))
        else:
            return int(len(text.split()) * 1.33)
    
    def extract_sections(self, markdown_content: str) -> List[Dict]:
        sections = []
        lines = markdown_content.split('\n')
        
        current_section = {
            'title': 'Introduction',
            'level': 1,
            'content': [],
            'start_line': 0
        }
        
        for i, line in enumerate(lines):
            if line.strip().startswith('#'):
                # save previous section
                if current_section['content']:
                    current_section['content'] = '\n'.join(current_section['content']).strip()
                    if current_section['content']:
                        sections.append(current_section.copy())
                
                # start new section
                header_level = len(line) - len(line.lstrip('#'))
                title = line.lstrip('#').strip()
                
                current_section = {
                    'title': title,
                    'level': header_level,
                    'content': [],
                    'start_line': i
                }
            else:
                # add content to current section
                if line.strip():  # skip empty lines
                    current_section['content'].append(line)
        
        # add final section
        if current_section['content']:
            current_section['content'] = '\n'.join(current_section['content']).strip()
            if current_section['content']:
                sections.append(current_section)
        
        return sections
    
    def filter_quality_chunks(self, chunks: List[Dict]) -> List[Dict]:
        """Remove low-quality chunks to improve RAG performance"""
        quality_chunks = []
        
        # Sections to skip (metadata, not research content)
        skip_sections = ['viewpoints', 'papers', 'affiliations', 'correspondence', 
                        'figures and tables', 'funding', 'conflicts', 'ethics']
        
        for chunk in chunks:
            content = chunk['content'].lower()
            section = chunk['metadata']['section'].lower()
            
            # Skip metadata sections
            if any(skip in section for skip in skip_sections):
                continue
            
            # Skip very short chunks (likely incomplete)
            if len(chunk['content'].split()) < 25:
                continue
            
            # Skip reference-heavy chunks
            if (content.count('doi:') > 3 or 
                content.count('et al') > 6 or
                content.count('pmid:') > 3 or
                content.count('http') > 4):
                continue
            
            # Skip chunks that are mostly author names/numbers
            words = chunk['content'].split()
            if (len(words) < 40 and 
                (sum(1 for w in words if w.isdigit()) > len(words) * 0.3 or
                 sum(1 for w in words if w[0].isupper() and len(w) > 2) > len(words) * 0.5)):
                continue
            
            quality_chunks.append(chunk)
        
        return quality_chunks
    
    def process_document(self, md_file_path: str, metadata_file_path: str = None) -> List[Dict]:
        
        with open(md_file_path, 'r', encoding='utf-8') as f:
            markdown_content = f.read()
        
        document_metadata = {'document_id': Path(md_file_path).stem, 'source_file': md_file_path}
        if metadata_file_path and os.path.exists(metadata_file_path):
            with open(metadata_file_path, 'r', encoding='utf-8') as f:
                loaded_metadata = json.load(f)
                document_metadata.update(loaded_metadata)
        
        sections = self.extract_sections(markdown_content)
        
        all_chunks = []
        global_chunk_index = 0 
        
        for section_idx, section in enumerate(sections):
            section_chunks = self.chunk_section(section, document_metadata, global_chunk_index, section_idx)
            all_chunks.extend(section_chunks)
            global_chunk_index += len(section_chunks)  
        
        for chunk in all_chunks:
            chunk['metadata']['total_chunks'] = len(all_chunks)
        
        return all_chunks

    def chunk_section(self, section: Dict, document_metadata: Dict, start_chunk_index: int = 0, section_idx: int = 0) -> List[Dict]:
        """Chunk a single section into optimal sizes"""
        content = section['content']
        section_title = section['title']
        
        if not content.strip():
            return []
        
        chunks = []
        
        paragraphs = [p.strip() for p in content.split('\n\n') if p.strip()]
        
        current_chunk = ""
        chunk_index = start_chunk_index 
        
        for paragraph in paragraphs:

            potential_chunk = current_chunk + "\n\n" + paragraph if current_chunk else paragraph
            token_count = self.count_tokens(potential_chunk)
            
            if token_count <= self.chunk_size:
                current_chunk = potential_chunk
            else:

                if current_chunk and self.count_tokens(current_chunk) >= self.min_chunk_size:
                    chunks.append(self._create_chunk(
                        current_chunk, 
                        chunk_index, 
                        section_title, 
                        document_metadata,
                        section_idx
                    ))
                    chunk_index += 1
                
                if self.count_tokens(paragraph) <= self.chunk_size:
                    current_chunk = paragraph
                else:

                    sentence_chunks = self._split_long_paragraph(paragraph, section_title, document_metadata, chunk_index, section_idx)
                    chunks.extend(sentence_chunks)
                    chunk_index += len(sentence_chunks)
                    current_chunk = ""
        
        if current_chunk and self.count_tokens(current_chunk) >= self.min_chunk_size:
            chunks.append(self._create_chunk(
                current_chunk, 
                chunk_index, 
                section_title, 
                document_metadata,
                section_idx
            ))
        
        return chunks

    def _create_chunk(self, content: str, chunk_index: int, section_title: str, document_metadata: Dict, section_idx: int = 0) -> Dict:

        # use both section and chunk index for unique IDs
        chunk_id = f"{document_metadata['document_id']}_sec{section_idx:02d}_chunk{chunk_index:03d}"
        
        return {
            'chunk_id': chunk_id,
            'content': content.strip(),
            'token_count': self.count_tokens(content),
            'metadata': {
                'document_id': document_metadata['document_id'],
                'section': section_title,
                'section_index': section_idx,
                'chunk_index': chunk_index,
                'source_file': document_metadata['source_file'],
                'paper_title': document_metadata.get('title', 'Unknown'),
                'authors': document_metadata.get('authors', []),
                'doi': document_metadata.get('doi', ''),
                'publication_date': document_metadata.get('publication_date', ''),
                'source_url': document_metadata.get('source_url', '')
            }
        }

    def _split_long_paragraph(self, paragraph: str, section_title: str, document_metadata: Dict, start_index: int, section_idx: int = 0) -> List[Dict]:
        """Split overly long paragraphs by sentences"""
        sentences = re.split(r'(?<=[.!?])\s+', paragraph)
        chunks = []
        current_chunk = ""
        chunk_index = start_index
        
        for sentence in sentences:
            potential_chunk = current_chunk + " " + sentence if current_chunk else sentence
            
            if self.count_tokens(potential_chunk) <= self.chunk_size:
                current_chunk = potential_chunk
            else:
                if current_chunk:
                    chunks.append(self._create_chunk(
                        current_chunk, 
                        chunk_index, 
                        section_title, 
                        document_metadata,
                        section_idx
                    ))
                    chunk_index += 1
                
                current_chunk = sentence
        
        if current_chunk:
            chunks.append(self._create_chunk(
                current_chunk, 
                chunk_index, 
                section_title, 
                document_metadata,
                section_idx
            ))
        
        return chunks
    
    def process_directory(self, input_dir: str, output_dir: str, max_files: int = None) -> Dict:

        input_path = Path(input_dir)
        output_path = Path(output_dir)
        output_path.mkdir(parents=True, exist_ok=True)
        
        md_files = list(input_path.glob("*.md"))
        if max_files:
            md_files = md_files[:max_files]
        
        all_chunks = []
        processing_stats = {
            'total_documents': len(md_files),
            'total_chunks': 0,
            'filtered_chunks': 0,
            'avg_tokens_per_chunk': 0,
            'sections_processed': 0
        }
        
        print(f"Processing {len(md_files)} markdown files...")
        
        for md_file in tqdm(md_files, desc="Chunking documents"):
            # Look for metadata file
            metadata_file = md_file.with_name(f"{md_file.stem}_processed_meta.json")
            metadata_path = metadata_file if metadata_file.exists() else None
            
            # Process document
            document_chunks = self.process_document(str(md_file), str(metadata_path) if metadata_path else None)
            all_chunks.extend(document_chunks)
            
            print(f"  {md_file.name}: {len(document_chunks)} chunks")
        
        print("Filtering low-quality chunks...")
        quality_chunks = self.filter_quality_chunks(all_chunks)
        
        chunks_file = output_path / "all_chunks.json"
        with open(chunks_file, 'w', encoding='utf-8') as f:
            json.dump(quality_chunks, f, indent=2, ensure_ascii=False)
        
        doc_index = {}
        for chunk in quality_chunks:
            doc_id = chunk['metadata']['document_id']
            if doc_id not in doc_index:
                doc_index[doc_id] = {
                    'chunk_count': 0,
                    'sections': set(),
                    'chunk_ids': []
                }
            doc_index[doc_id]['chunk_count'] += 1
            doc_index[doc_id]['sections'].add(chunk['metadata']['section'])
            doc_index[doc_id]['chunk_ids'].append(chunk['chunk_id'])
        
        for doc_id in doc_index:
            doc_index[doc_id]['sections'] = list(doc_index[doc_id]['sections'])
        
        index_file = output_path / "document_index.json"
        with open(index_file, 'w', encoding='utf-8') as f:
            json.dump(doc_index, f, indent=2)
        
        processing_stats['total_chunks'] = len(all_chunks)
        processing_stats['filtered_chunks'] = len(quality_chunks)
        if quality_chunks:
            processing_stats['avg_tokens_per_chunk'] = sum(c['token_count'] for c in quality_chunks) / len(quality_chunks)
        
        stats_file = output_path / "chunking_stats.json"
        with open(stats_file, 'w', encoding='utf-8') as f:
            json.dump(processing_stats, f, indent=2)
        
        print(f"\n{'='*60}")
        print("CHUNKING COMPLETE")
        print(f"{'='*60}")
        print(f"Total documents: {processing_stats['total_documents']}")
        print(f"Raw chunks: {processing_stats['total_chunks']}")
        print(f"Quality chunks: {processing_stats['filtered_chunks']}")
        print(f"Filtering efficiency: {(processing_stats['filtered_chunks']/processing_stats['total_chunks']*100):.1f}%")
        print(f"Average tokens per chunk: {processing_stats['avg_tokens_per_chunk']:.1f}")
        print(f"Output saved to: {chunks_file}")
        print(f"Document index: {index_file}")
        
        return processing_stats

# Usage
if __name__ == "__main__":
    chunker = DocumentChunker(
        chunk_size=500,  # Target chunk size in tokens
        chunk_overlap=50,  # Overlap between chunks
        min_chunk_size=100  # Minimum chunk size
    )
    
    input_directory = "D:/PsyWiz/processed_md"
    output_directory = "D:/PsyWiz/chunks"
    
    # For full processing, remove max_files limit
    stats = chunker.process_directory(
        input_directory, 
        output_directory, 
        max_files=None  # Process all files
    )

Processing 2 markdown files...


Chunking documents:  50%|█████     | 1/2 [00:00<00:00,  9.09it/s]

  article_2.md: 36 chunks


Chunking documents: 100%|██████████| 2/2 [00:00<00:00,  7.48it/s]

  article_3.md: 68 chunks
Filtering low-quality chunks...

CHUNKING COMPLETE
Total documents: 2
Raw chunks: 104
Quality chunks: 87
Filtering efficiency: 83.7%
Average tokens per chunk: 305.0
Output saved to: D:\PsyWiz\chunks\all_chunks.json
Document index: D:\PsyWiz\chunks\document_index.json
